In [ ]:
!pip install osmnx geopy

In [ ]:
# ==============================
# MODULE 1: Data Collection (Multiple Cities)
# ==============================

import requests
import pandas as pd
from datetime import datetime
import osmnx as ox
import os
from geopy.geocoders import Nominatim

API_KEY = "bb12516604fd2a9fb10e5b4a62905356"

print("Starting Module 1: Multi-City Data Collection\n")

# ==============================
# MULTIPLE CITY INPUT
# ==============================
cities_input = input("Enter cities (comma-separated): ")
cities = [city.strip() for city in cities_input.split(",")]

geolocator = Nominatim(user_agent="pollution_app")

for CITY in cities:
    print(f"\n🌍 Processing: {CITY}")

    try:
        location = geolocator.geocode(CITY + ", India")

        if location is None:
            print(f"❌ {CITY} not found. Skipping...")
            continue

        LAT = location.latitude
        LON = location.longitude
        timestamp = datetime.utcnow().isoformat()

        print(f"📍 {CITY}: ({LAT}, {LON})")

    except Exception as e:
        print(f"Geocoding Error for {CITY}:", e)
        continue

    # ==============================
    # WEATHER DATA
    # ==============================
    try:
        url = "https://api.openweathermap.org/data/2.5/weather"
        params = {"lat": LAT, "lon": LON, "appid": API_KEY, "units": "metric"}

        res = requests.get(url, params=params, timeout=10)
        data = res.json()

        weather = {
            "city": CITY,
            "latitude": LAT,
            "longitude": LON,
            "temperature_C": data["main"]["temp"],
            "humidity_%": data["main"]["humidity"],
            "wind_speed_mps": data["wind"]["speed"],
            "timestamp": timestamp
        }

        pd.DataFrame([weather]).to_csv(
            "/content/weather_data.csv",
            mode="a",
            header=not os.path.exists("/content/weather_data.csv"),
            index=False
        )

        print("✔ Weather saved")

    except Exception as e:
        print(f"Weather Error ({CITY}):", e)
        continue

    # ==============================
    # AIR QUALITY
    # ==============================
    try:
        url = "https://api.openweathermap.org/data/2.5/air_pollution"
        params = {"lat": LAT, "lon": LON, "appid": API_KEY}

        res = requests.get(url, params=params, timeout=10)
        data = res.json()

        comp = data["list"][0]["components"]

        air = {
            "city": CITY,
            "latitude": LAT,
            "longitude": LON,
            "PM2_5": comp.get("pm2_5"),
            "PM10": comp.get("pm10"),
            "NO2": comp.get("no2"),
            "CO": comp.get("co"),
            "SO2": comp.get("so2"),
            "O3": comp.get("o3"),
            "timestamp": timestamp
        }

        pd.DataFrame([air]).to_csv(
            "/content/air_quality_data.csv",
            mode="a",
            header=not os.path.exists("/content/air_quality_data.csv"),
            index=False
        )

        print("✔ Air Quality saved")

    except Exception as e:
        print(f"Air Error ({CITY}):", e)
        continue

    # ==============================
    # FINAL DATASET
    # ==============================
    try:
        weather = pd.read_csv("/content/weather_data.csv").tail(1)
        air = pd.read_csv("/content/air_quality_data.csv").tail(1)

        final_df = pd.concat([weather, air], axis=1)
        final_df = final_df.loc[:, ~final_df.columns.duplicated()]

        final_df.to_csv(
            "/content/final_dataset.csv",
            mode="a",
            header=not os.path.exists("/content/final_dataset.csv"),
            index=False
        )

        print("✅ Final dataset appended")

    except Exception as e:
        print(f"Final Dataset Error ({CITY}):", e)

print("\n🎉 Module 1 Multi-City Completed!")

Starting Module 1: Multi-City Data Collection

Enter cities (comma-separated): Kollam

🌍 Processing: Kollam


/tmp/ipykernel_15169/302910496.py:36: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat()


📍 Kollam: (8.8870533, 76.5906696)
✔ Weather saved
✔ Air Quality saved
✅ Final dataset appended

🎉 Module 1 Multi-City Completed!


In [ ]:
# ==============================
# MODULE 2: Data Cleaning (Append Mode)
# ==============================

import pandas as pd
import os

print("Starting Module 2: Data Cleaning\n")

try:
    # ==============================
    # LOAD RAW DATA
    # ==============================
    input_path = "/content/final_dataset.csv"

    if not os.path.exists(input_path):
        raise Exception("❌ final_dataset.csv not found. Run Module 1 first.")

    df = pd.read_csv(input_path)

    print(f"📊 Loaded Data Shape: {df.shape}")

    # ==============================
    # REMOVE DUPLICATES
    # ==============================
    before = len(df)
    df = df.drop_duplicates()
    after = len(df)
    print(f"✔ Removed {before - after} duplicate rows")

    # ==============================
    # HANDLE MISSING VALUES
    # ==============================
    df = df.fillna(method='ffill').fillna(method='bfill')

    # ==============================
    # NORMALIZE POLLUTANTS
    # ==============================
    pollutant_cols = ["PM2_5", "PM10", "NO2", "CO", "SO2", "O3"]

    for col in pollutant_cols:
        if col in df.columns:
            if df[col].max() != df[col].min():  # avoid divide by zero
                df[col] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())

    print("✔ Normalization completed")

    # ==============================
    # TIME FEATURE ENGINEERING
    # ==============================
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

    df['hour'] = df['timestamp'].dt.hour
    df['day'] = df['timestamp'].dt.day
    df['month'] = df['timestamp'].dt.month

    print("✔ Time features added")

    # ==============================
    # INDUSTRY FEATURE: AIR QUALITY INDEX (SIMPLE SCORE)
    # ==============================
    df['AQI_score'] = df[pollutant_cols].mean(axis=1)

    print("✔ AQI score calculated")

    # ==============================
    # SAVE CLEANED DATA (APPEND MODE)
    # ==============================
    output_path = "/content/cleaned_dataset.csv"

    df.to_csv(
        output_path,
        mode="a",   # 🔥 APPEND MODE
        header=not os.path.exists(output_path),
        index=False
    )

    print(f"✅ Cleaned data appended → cleaned_dataset.csv")

except Exception as e:
    print("❌ Error:", e)

print("\n🎉 Module 2 Completed Successfully!")

Starting Module 2: Data Cleaning

📊 Loaded Data Shape: (419, 13)
✔ Removed 0 duplicate rows
✔ Normalization completed
✔ Time features added
✔ AQI score calculated
✅ Cleaned data appended → cleaned_dataset.csv

🎉 Module 2 Completed Successfully!


/tmp/ipykernel_15169/570112084.py:34: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='ffill').fillna(method='bfill')


In [ ]:
# ==============================
# MODULE 3: FINAL SMART APPEND SYSTEM (FIXED)
# ==============================

import pandas as pd
import os

print("Starting Module 3: Smart Append Classification\n")

try:
    input_path = "/content/cleaned_dataset.csv"

    if not os.path.exists(input_path):
        raise Exception("❌ cleaned_dataset.csv not found.")

    df = pd.read_csv(input_path)

    print(f"📊 New Data Loaded: {df.shape}")

    # ==============================
    # DOMAIN CLASSIFICATION
    # ==============================
    def classify_domain(row):
        try:
            if row["NO2"] > 0.7 and row["CO"] > 0.6:
                return "Vehicular"
            elif row["SO2"] > 0.6:
                return "Industrial"
            elif row["PM2_5"] > 0.8 and row["PM10"] > 0.8:
                return "Biomass Burning"
            elif row["PM2_5"] > 0.6:
                return "Agricultural"
            elif row["PM10"] > 0.6:
                return "Urban Dust"
            elif row["humidity_%"] > 85 and row["wind_speed_mps"] < 2:
                return "Natural"
            else:
                pollutants = {
                    "Vehicular": row["NO2"] + row["CO"],
                    "Industrial": row["SO2"],
                    "Agricultural": row["PM2_5"],
                    "Urban Dust": row["PM10"]
                }
                return max(pollutants, key=pollutants.get)
        except:
            return "Vehicular"

    # ==============================
    # CAUSE TYPE
    # ==============================
    def classify_cause(row):
        try:
            if row["PM2_5"] > 0.8 and row["CO"] > 0.6:
                return "Burning"
            elif row["NO2"] > 0.6 or row["SO2"] > 0.6:
                return "Human Activity"
            elif row["humidity_%"] > 85 and row["wind_speed_mps"] < 2:
                return "Natural"
            else:
                return "Human Activity"
        except:
            return "Human Activity"

    # ==============================
    # CONFIDENCE SCORE
    # ==============================
    def calculate_confidence(row):
        try:
            values = [
                row["PM2_5"], row["PM10"], row["NO2"],
                row["CO"], row["SO2"], row["O3"]
            ]
            total = sum(values)
            if total == 0:
                return 0.0
            return round(max(values) / total, 2)
        except:
            return 0.0

    # ==============================
    # APPLY
    # ==============================
    df["source_domain"] = df.apply(classify_domain, axis=1)
    df["cause_type"] = df.apply(classify_cause, axis=1)
    df["confidence"] = df.apply(calculate_confidence, axis=1)

    print("✔ Classification done")

    # ==============================
    # SMART APPEND SYSTEM (FIXED)
    # ==============================
    output_path = "/content/labeled_dataset.csv"

    required_cols = [
        "city","latitude","longitude","temperature_C","humidity_%",
        "wind_speed_mps","timestamp","PM2_5","PM10","NO2","CO","SO2","O3",
        "hour","day","month","AQI_score",
        "source_domain","cause_type","confidence"
    ]

    # Ensure schema consistency
    df = df.reindex(columns=required_cols)

    # 🔥 DEBUG: Check files
    print("📂 Files in /content:", os.listdir("/content"))

    if os.path.isfile(output_path):
        print("✔ Previous file found → appending")

        old_df = pd.read_csv(output_path)
        old_df = old_df.reindex(columns=required_cols)

        final_df = pd.concat([old_df, df], ignore_index=True)

        # Remove duplicates
        final_df = final_df.drop_duplicates(subset=["city", "timestamp"])

    else:
        print("⚠ No previous file → creating new dataset")
        final_df = df

    # Save final dataset
    final_df.to_csv(output_path, index=False)

    print("✅ Final dataset saved successfully")
    print("📊 Total rows after append:", final_df.shape)

except Exception as e:
    print("❌ Error:", e)

print("\n🎉 Module 3 Completed Successfully!")

Starting Module 3: Smart Append Classification

📊 New Data Loaded: (5142, 17)
✔ Classification done
📂 Files in /content: ['.config', 'labeled_dataset.csv', 'pollution_model.pkl', 'weather_data.csv', 'cleaned_dataset.csv', 'air_quality_data.csv', 'final_dataset.csv', 'sample_data']
✔ Previous file found → appending
✅ Final dataset saved successfully
📊 Total rows after append: (419, 20)

🎉 Module 3 Completed Successfully!


In [ ]:
# ==============================
# MODULE 4: Model Training (UPDATED)
# ==============================

import pandas as pd
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

print("Starting Module 4: Model Training\n")

try:
    input_path = "/content/labeled_dataset.csv"

    if not os.path.exists(input_path):
        raise Exception("❌ labeled_dataset.csv not found.")

    df = pd.read_csv(input_path)

    print(f"📊 Dataset Shape: {df.shape}")

    # ==============================
    # REMOVE DUPLICATES
    # ==============================
    df = df.drop_duplicates()

    # ==============================
    # FEATURES & TARGET
    # ==============================
    features = ["PM2_5", "PM10", "NO2", "CO", "SO2", "O3", "AQI_score"]

    # Ensure columns exist
    features = [col for col in features if col in df.columns]

    X = df[features]
    y = df["source_domain"]   # 🔥 FIXED HERE

    # Remove missing
    mask = y.notna()
    X = X[mask]
    y = y[mask]

    # ==============================
    # HANDLE SMALL DATA CASE
    # ==============================
    if len(df) < 5:
        print("⚠️ Not enough data, training on full dataset")

        model = RandomForestClassifier()
        model.fit(X, y)

    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        model = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            random_state=42
        )

        model.fit(X_train, y_train)

        preds = model.predict(X_test)

        print("\n📊 Model Evaluation:")
        print("Accuracy:", accuracy_score(y_test, preds))
        print(classification_report(y_test, preds))

    # ==============================
    # SAVE MODEL
    # ==============================
    joblib.dump(model, "/content/pollution_model.pkl")

    print("✅ Model saved → pollution_model.pkl")

except Exception as e:
    print("❌ Error:", e)

print("\n🎉 Module 4 Completed Successfully!")

Starting Module 4: Model Training

📊 Dataset Shape: (419, 20)

📊 Model Evaluation:
Accuracy: 0.7976190476190477
              precision    recall  f1-score   support

Agricultural       0.50      0.38      0.43         8
  Industrial       0.80      0.80      0.80         5
     Natural       0.00      0.00      0.00         2
  Urban Dust       0.81      0.71      0.76        24
   Vehicular       0.83      0.96      0.89        45

    accuracy                           0.80        84
   macro avg       0.59      0.57      0.57        84
weighted avg       0.77      0.80      0.78        84

✅ Model saved → pollution_model.pkl

🎉 Module 4 Completed Successfully!


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# ==============================
# MODULE 5: Geospatial Heatmap
# ==============================

import pandas as pd
import folium
from folium.plugins import HeatMap
import os

print("Starting Module 5: Geospatial Visualization\n")

try:
    # ==============================
    # LOAD DATA
    # ==============================
    input_path = "/content/labeled_dataset.csv"

    if not os.path.exists(input_path):
        raise Exception("❌ labeled_dataset.csv not found. Run Module 3 first.")

    df = pd.read_csv(input_path)

    print(f"📊 Loaded Data Shape: {df.shape}")

    # Remove duplicates (important for maps)
    df = df.drop_duplicates()

    # ==============================
    # MAP INITIALIZATION
    # ==============================
    center_lat = df["latitude"].mean()
    center_lon = df["longitude"].mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

    # ==============================
    # HEATMAP (PM2.5 intensity)
    # ==============================
    heat_data = df[["latitude", "longitude", "PM2_5"]].dropna().values.tolist()

    HeatMap(
        heat_data,
        radius=15,
        blur=10
    ).add_to(m)

    print("✔ Heatmap layer added")

    # ==============================
    # SOURCE MARKERS (INDUSTRY FEATURE)
    # ==============================
    def get_color(source):
        if source == "Vehicular":
            return "blue"
        elif source == "Industrial":
            return "red"
        elif source == "Agricultural":
            return "green"
        elif source == "Natural":
            return "purple"
        else:
            return "gray"

    for _, row in df.iterrows():
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=5,
            color=get_color(row["source_domain"]),
            fill=True,
            fill_opacity=0.7,
            popup=f"""
            City: {row.get('city','')}
            Source: {row.get('source_domain','')}
            AQI Score: {row.get('AQI_score','')}
            """
        ).add_to(m)

    print("✔ Source markers added")

    # ==============================
    # SAVE MAP
    # ==============================
    output_path = "/content/pollution_heatmap.html"
    m.save(output_path)

    print(f"✅ Map saved → pollution_heatmap.html")

except Exception as e:
    print("❌ Error:", e)

print("\n🎉 Module 5 Completed Successfully!")

Starting Module 5: Geospatial Visualization

📊 Loaded Data Shape: (10, 20)
✔ Heatmap layer added
✔ Source markers added
✅ Map saved → pollution_heatmap.html

🎉 Module 5 Completed Successfully!


In [ ]:
!kill -9 -1

In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 118.0 MB/s eta 0:00:00


In [ ]:
%%writefile app.py

# ==============================



# MODULE 6: ADVANCED DASHBOARD
# ==============================



import streamlit as st
import pandas as pd
import joblib
import os

st.set_page_config(page_title="EnviroScan Dashboard", layout="wide")

st.title("🌍 EnviroScan: Pollution Source Identifier")

# ==============================
# LOAD DATA
# ==============================
data_path = "/content/labeled_dataset.csv"
model_path = "/content/pollution_model.pkl"

if not os.path.exists(data_path):
    st.error("❌ Dataset not found. Run previous modules.")
    st.stop()

df = pd.read_csv(data_path)

if not os.path.exists(model_path):
    st.error("❌ Model not found. Run Module 4.")
    st.stop()

model = joblib.load(model_path)

# ==============================
# SIDEBAR FILTERS
# ==============================
st.sidebar.header("🔎 Filters")

city_filter = st.sidebar.selectbox(
    "Select City",
    options=df["city"].dropna().unique()
)

filtered_df = df[df["city"] == city_filter]

# ==============================
# KPI METRICS
# ==============================
col1, col2, col3 = st.columns(3)

col1.metric("📊 Avg PM2.5", round(filtered_df["PM2_5"].mean(), 2))
col2.metric("🚗 Dominant Source", filtered_df["source_domain"].mode()[0])
col3.metric("⚠️ Avg AQI Score", round(filtered_df["AQI_score"].mean(), 2))

# ==============================
# ALERT SYSTEM 🚨
# ==============================
st.subheader("🚨 Pollution Alerts")

latest = filtered_df.tail(1)

if latest["AQI_score"].values[0] > 0.7:
    st.error("⚠️ HIGH POLLUTION ALERT!")
elif latest["AQI_score"].values[0] > 0.4:
    st.warning("⚠️ Moderate Pollution")
else:
    st.success("✅ Air Quality Normal")

# ==============================
# DATA TABLE
# ==============================
st.subheader("📋 Recent Data")
st.dataframe(filtered_df.tail(10))

# ==============================
# CHARTS
# ==============================
st.subheader("📈 Pollution Trends")

st.line_chart(filtered_df[["PM2_5", "PM10", "NO2"]])

st.subheader("📊 Source Distribution")

st.bar_chart(filtered_df["source_domain"].value_counts())

# ==============================
# PREDICTION SECTION 🔮
# ==============================
st.subheader("🔮 Predict Pollution Source")

col1, col2, col3 = st.columns(3)

pm25 = col1.slider("PM2.5", 0.0, 1.0)
pm10 = col2.slider("PM10", 0.0, 1.0)
no2 = col3.slider("NO2", 0.0, 1.0)

col4, col5, col6 = st.columns(3)

co = col4.slider("CO", 0.0, 1.0)
so2 = col5.slider("SO2", 0.0, 1.0)
o3 = col6.slider("O3", 0.0, 1.0)

aqi = (pm25 + pm10 + no2 + co + so2 + o3) / 6

if st.button("Predict Source"):
    pred = model.predict([[pm25, pm10, no2, co, so2, o3, aqi]])
    st.success(f"🌍 Predicted Source: {pred[0]}")

# ==============================
# DOWNLOAD OPTION
# ==============================
# ==============================
st.subheader("⬇ Download Data")

st.download_button(
    label="Download Dataset",
    data=filtered_df.to_csv(index=False),
    file_name="pollution_data.csv",
    mime="text/csv"
)

# ==============================
# FOOTER
# ==============================
st.markdown("---")
st.caption("🚀 EnviroScan Project | AI + Geospatial Analytics")

Writing app.py


In [ ]:
!streamlit run app.py --server.port 8501 & npx localtunnel --port 8501 --yes

⠙⠹⠸

⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.6.228.160:8501

y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹your url is: https://pink-mugs-brush.loca.lt
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
  Stopping...
^C


In [ ]:
import pandas as pd

df = pd.read_csv("/content/labeled_dataset.csv")
df.head()

,city,latitude,longitude,temperature_C,humidity_%,wind_speed_mps,timestamp,PM2_5,PM10,NO2,CO,SO2,O3,hour,day,month,AQI_score,source_domain,cause_type,confidence
0,Diphu,25.837194,93.438366,19.27,89,0.84,2026-03-22 03:21:36.703161,0.190765,0.182323,0.083267,0.278300,0.078141,0.462851,3,22,3,0.212608,Natural,Natural,0.36
1,Hojai,25.962940,92.981829,20.11,84,1.88,2026-03-22 03:21:38.240370,0.253098,0.246019,0.178430,0.309639,0.119324,0.648774,3,22,3,0.292547,Vehicular,Human Activity,0.37
2,Wokha,26.242251,94.192115,17.11,97,0.53,2026-03-22 03:21:38.916071,0.062334,0.051472,0.000000,0.145927,0.003168,0.928902,3,22,3,0.198634,Natural,Natural,0.78
3,Mon,26.666667,95.000000,15.76,99,1.17,2026-03-22 03:21:40.194479,0.003856,0.000000,0.005551,0.098260,0.003168,0.717206,3,22,3,0.138007,Natural,Natural,0.87
4,Tuensang,26.234211,94.741553,12.86,99,0.96,2026-03-22 03:21:41.135506,0.034977,0.027344,0.010309,0.126687,0.000000,0.794881,3,22,3,0.165700,Natural,Natural,0.80


In [ ]:
df.tail(20)

,city,latitude,longitude,temperature_C,humidity_%,wind_speed_mps,timestamp,PM2_5,PM10,NO2,CO,SO2,O3,hour,day,month,AQI_score,source_domain,cause_type,confidence
399,Fatehpur,25.834802,80.915455,30.00,20,5.37,2026-03-22 05:37:11.040491,0.382631,0.374654,0.044841,0.176463,0.138816,0.793032,5,22,3,0.318406,Agricultural,Human Activity,0.42
400,Unnao,26.567326,80.619819,29.63,22,4.95,2026-03-22 05:37:11.629672,0.584033,0.577406,0.061820,0.240957,0.186352,0.797661,5,22,3,0.408038,Agricultural,Human Activity,0.33
401,Rae Bareli,26.250000,81.250000,29.30,23,4.80,2026-03-22 05:37:12.136663,0.513434,0.507553,0.055725,0.218535,0.160980,0.783713,5,22,3,0.373323,Agricultural,Human Activity,0.35
402,Sultanpur,26.260473,82.238053,27.94,35,2.42,2026-03-22 05:37:12.644084,0.600662,0.605347,0.094036,0.259577,0.282006,0.815362,5,22,3,0.442832,Agricultural,Human Activity,0.31
403,Ambedkar Nagar,26.404594,82.606515,27.39,39,2.06,2026-03-22 05:37:13.174414,0.562399,0.565705,0.108838,0.288305,0.251677,0.764136,5,22,3,0.423510,Urban Dust,Human Activity,0.30
404,Balrampur,27.447699,82.395624,27.40,37,3.71,2026-03-22 05:37:13.670533,0.757025,0.747394,0.100566,0.406387,0.226888,0.775269,5,22,3,0.502255,Agricultural,Human Activity,0.26
405,Siddharthnagar,27.251172,82.871653,27.28,37,3.24,2026-03-22 05:37:14.151615,0.351605,0.334586,0.047018,0.253261,0.080198,0.677571,5,22,3,0.290707,Agricultural,Human Activity,0.39
406,Sant Kabir Nagar,26.755082,83.039855,27.08,39,2.59,2026-03-22 05:37:14.594503,0.423358,0.417772,0.097083,0.270918,0.188101,0.723980,5,22,3,0.353536,Agricultural,Human Activity,0.34
407,Mau,26.039091,83.498585,27.02,38,2.73,2026-03-22 05:37:15.056561,0.574794,0.585419,0.202003,0.361667,0.365704,0.802977,5,22,3,0.482094,Urban Dust,Human Activity,0.28
408,Azamgarh,26.023639,83.029310,27.44,38,2.48,2026-03-22 05:37:15.516848,0.523597,0.532232,0.139312,0.291011,0.285506,0.778521,5,22,3,0.425030,Urban Dust,Human Activity,0.31


In [ ]:
print(df.shape)

(211, 20)


In [ ]:
from IPython.display import IFrame

IFrame("/content/pollution_heatmap.html", width=800, height=500)

In [ ]:
import pandas as pd

df = pd.read_csv("/content/labeled_dataset.csv")
df

,city,latitude,longitude,temperature_C,humidity_%,wind_speed_mps,timestamp,PM2_5,PM10,NO2,CO,SO2,O3,hour,day,month,AQI_score,source_domain,cause_type,confidence
0,Diphu,25.837194,93.438366,19.27,89,0.84,2026-03-22 03:21:36.703161,0.190765,0.182323,0.083267,0.278300,0.078141,0.462851,3,22,3,0.212608,Natural,Natural,0.36
1,Hojai,25.962940,92.981829,20.11,84,1.88,2026-03-22 03:21:38.240370,0.253098,0.246019,0.178430,0.309639,0.119324,0.648774,3,22,3,0.292547,Vehicular,Human Activity,0.37
2,Wokha,26.242251,94.192115,17.11,97,0.53,2026-03-22 03:21:38.916071,0.062334,0.051472,0.000000,0.145927,0.003168,0.928902,3,22,3,0.198634,Natural,Natural,0.78
3,Mon,26.666667,95.000000,15.76,99,1.17,2026-03-22 03:21:40.194479,0.003856,0.000000,0.005551,0.098260,0.003168,0.717206,3,22,3,0.138007,Natural,Natural,0.87
4,Tuensang,26.234211,94.741553,12.86,99,0.96,2026-03-22 03:21:41.135506,0.034977,0.027344,0.010309,0.126687,0.000000,0.794881,3,22,3,0.165700,Natural,Natural,0.80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
414,Lalitpur,24.699969,78.518620,27.77,27,1.84,2026-03-22 05:37:18.204638,0.165063,0.183178,0.047453,0.106191,0.106445,0.706467,5,22,3,0.219133,Urban Dust,Human Activity,0.54
415,Jalaun,26.104863,79.391914,28.27,28,3.34,2026-03-22 05:37:18.620640,0.388175,0.393022,0.083152,0.194476,0.340041,0.809982,5,22,3,0.368141,Urban Dust,Human Activity,0.37
416,Kasganj,27.788853,78.822970,29.00,26,4.47,2026-03-22 05:37:19.013439,0.632920,0.660308,0.141924,0.363236,0.233596,0.788967,5,22,3,0.470159,Agricultural,Human Activity,0.28
417,Hathras,27.572432,78.111621,27.80,31,3.97,2026-03-22 05:37:19.412719,0.317884,0.355790,0.084023,0.223551,0.044036,0.722730,5,22,3,0.291336,Urban Dust,Human Activity,0.41


In [ ]:
import pandas as pd

df = pd.read_csv("/content/labeled_dataset.csv")

# Keep only one entry per city (remove extra duplicates)
df = df.drop_duplicates(subset=["city"], keep="first")

# Save
df.to_csv("/content/labeled_dataset.csv", index=False)

print("✅ Removed duplicate cities (kept one entry)")
print("📊 New shape:", df.shape)

✅ Removed duplicate cities (kept one entry)
📊 New shape: (400, 20)


In [ ]:
import pandas as pd

df = pd.read_csv("/content/labeled_dataset.csv")
df

,city,latitude,longitude,temperature_C,humidity_%,wind_speed_mps,timestamp,PM2_5,PM10,NO2,CO,SO2,O3,hour,day,month,AQI_score,source_domain,cause_type,confidence
0,Diphu,25.837194,93.438366,19.27,89,0.84,2026-03-22 03:21:36.703161,0.190765,0.182323,0.083267,0.278300,0.078141,0.462851,3,22,3,0.212608,Natural,Natural,0.36
1,Hojai,25.962940,92.981829,20.11,84,1.88,2026-03-22 03:21:38.240370,0.253098,0.246019,0.178430,0.309639,0.119324,0.648774,3,22,3,0.292547,Vehicular,Human Activity,0.37
2,Wokha,26.242251,94.192115,17.11,97,0.53,2026-03-22 03:21:38.916071,0.062334,0.051472,0.000000,0.145927,0.003168,0.928902,3,22,3,0.198634,Natural,Natural,0.78
3,Mon,26.666667,95.000000,15.76,99,1.17,2026-03-22 03:21:40.194479,0.003856,0.000000,0.005551,0.098260,0.003168,0.717206,3,22,3,0.138007,Natural,Natural,0.87
4,Tuensang,26.234211,94.741553,12.86,99,0.96,2026-03-22 03:21:41.135506,0.034977,0.027344,0.010309,0.126687,0.000000,0.794881,3,22,3,0.165700,Natural,Natural,0.80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,Batlagundu,10.163609,77.759098,33.68,35,2.65,2026-03-22 07:01:42.098116,0.136192,0.135380,0.049978,0.108956,0.052806,0.600818,7,22,3,0.180688,Vehicular,Human Activity,0.55
396,Tiruchendur,8.495568,78.123324,30.74,55,5.23,2026-03-22 07:01:42.895857,0.085380,0.078363,0.015645,0.085361,0.012590,0.449636,7,22,3,0.121163,Vehicular,Human Activity,0.62
397,Alangudi,10.360370,78.977872,33.23,40,3.12,2026-03-22 07:01:45.519704,0.042728,0.042337,0.043025,0.074260,0.042315,0.350051,7,22,3,0.099119,Vehicular,Human Activity,0.59
398,Keeranur,10.573386,78.784265,33.20,38,2.83,2026-03-22 07:01:46.386630,0.061283,0.063045,0.048240,0.082951,0.038993,0.433339,7,22,3,0.121308,Vehicular,Human Activity,0.60


In [ ]:
from google.colab import files
files.download("/content/labeled_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>